In [1]:
"""
Walk-forward + multi-run block-bootstrap, saving model & scaler
( filenames include the ticker so they NEVER overwrite each other )

In-sample  : 2000-01-01 → 2022-12-30
Out-sample : 2023-01-03 → today
"""

import math, random, pickle
import numpy as np, pandas as pd, yfinance as yf
import torch, torch.nn as nn, torch.optim as optim
from sklearn.model_selection import TimeSeriesSplit

# ───── CONFIG ────────────────────────────────────────────────────────────
base_seed     = 100
ticker        = "FBK"
sector_ticker = "XLF"
start_date    = "2000-01-01"

seq_len       = 5
hidden_size   = 512
num_layers    = 1
batch_size    = 64
epochs        = 100
threshold     = 0.7
n_splits      = 5
num_runs      = 5
device        = torch.device("cpu")

N_BOOT     = 10_000      # block-bootstrap
BLOCK_DAYS = 10

# → files include the ticker
MODEL_FILE  = f"best_model_{ticker}.pth"
SCALER_FILE = f"scaler_{ticker}.pkl"

# ───── download & prep ───────────────────────────────────────────────────
df = yf.download([ticker, sector_ticker],
                 start=start_date, interval="1d", group_by="ticker")
if isinstance(df.columns, pd.MultiIndex):
    df.columns = ["_".join(c).strip() for c in df.columns.values]
df.dropna(inplace=True)

feat_cols  = [f"{ticker}_{c}" for c in ["Open","High","Low","Close","Volume"]]
feat_cols += [f"{sector_ticker}_{c}" for c in
              ["Open","High","Low","Close","Volume"]
              if f"{sector_ticker}_{c}" in df.columns]

df["Target"] = (df[f"{ticker}_Close"] > df[f"{ticker}_Open"]).astype(int)

def make_seq(data, L):
    X, y = [], []
    for i in range(len(data) - L):
        X.append(data[feat_cols].iloc[i:i+L].values)
        y.append(data["Target"].iloc[i+L])
    return np.array(X), np.array(y)

X_raw, y = make_seq(df, seq_len)
tscv = TimeSeriesSplit(n_splits=n_splits)

# ───── block-bootstrap helper ────────────────────────────────────────────
def block_bootstrap(series: np.ndarray, blk: int, iters: int) -> float:
    T, k = len(series), math.ceil(len(series) / blk)
    means = []
    for _ in range(iters):
        starts = np.random.randint(0, T-blk+1, size=k)
        sample = np.concatenate([series[s:s+blk] for s in starts])[:T]
        means.append(sample.mean())
    means = np.array(means)
    return (np.sum(means <= 0) + 1) / (iters + 1)

# ───── model class ───────────────────────────────────────────────────────
class LSTMClassifier(nn.Module):
    def __init__(self, in_size, hsz=hidden_size, nlayers=num_layers):
        super().__init__()
        self.lstm = nn.LSTM(in_size, hsz, nlayers,
                            batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(hsz, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return torch.sigmoid(self.fc(h[-1])).squeeze()

# ───── runs ──────────────────────────────────────────────────────────────
all_diffs_final, best_avg_up, best_run = [], -np.inf, None
for run in range(1, num_runs+1):
    seed = base_seed + run
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    print(f"\n=== Run {run}/{num_runs} (seed={seed}) ===")

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_raw), 1):
        X_tr_raw, y_tr = X_raw[tr_idx], y[tr_idx]
        X_te_raw, y_te = X_raw[te_idx], y[te_idx]

        flat = X_tr_raw.reshape(-1, X_tr_raw.shape[-1])
        mu, sig = flat.mean(0), flat.std(0); sig[sig==0] = 1
        X_tr = (X_tr_raw - mu)/sig
        X_te = (X_te_raw - mu)/sig

        model = LSTMClassifier(len(feat_cols)).to(device)
        opt   = optim.Adam(model.parameters(), lr=1e-3)
        loss  = nn.BCELoss()

        for _ in range(epochs):
            perm = torch.randperm(len(X_tr))
            for i in range(0, len(X_tr), batch_size):
                idx = perm[i:i+batch_size]
                xb  = torch.tensor(X_tr[idx], dtype=torch.float32)
                yb  = torch.tensor(y_tr[idx], dtype=torch.float32)
                opt.zero_grad(); loss(model(xb), yb).backward(); opt.step()

        if fold == n_splits:                             # ← final fold only
            with torch.no_grad():
                probs = model(torch.tensor(X_te, dtype=torch.float32)).numpy()
            mask = probs >= threshold
            tmp  = df.iloc[te_idx+seq_len].copy()
            tmp["Ret"] = (df.iloc[te_idx+seq_len][f"{ticker}_Close"] -
                          df.iloc[te_idx+seq_len][f"{ticker}_Open"]) / \
                         df.iloc[te_idx+seq_len][f"{ticker}_Open"]

            overall_mean = tmp["Ret"].mean()
            diffs_final  = tmp.loc[mask, "Ret"].values - overall_mean
            all_diffs_final.append(diffs_final)

            avg_up = tmp.loc[mask, "Ret"].mean() if mask.any() else np.nan
            cov    = mask.mean()*100
            print(f"[Final Fold] Cov={cov:.1f}%  AvgUp={avg_up:.4%}")

            if mask.any() and avg_up > best_avg_up:
                best_avg_up = avg_up; best_run = run
                torch.save(model.state_dict(), MODEL_FILE)
                with open(SCALER_FILE, "wb") as f:
                    pickle.dump((mu, sig), f)
                print(f"  → Saved best model to {MODEL_FILE}")

# ───── pooled bootstrap & summary ────────────────────────────────────────
pooled = np.concatenate(all_diffs_final)
p_val  = block_bootstrap(pooled, BLOCK_DAYS, N_BOOT)

print(f"\nPooled final-fold block-bootstrap p-value = {p_val:.4f}")
print(f"Best run #{best_run}  AvgUp = {best_avg_up:.4%}")
print(f"Saved files →  {MODEL_FILE} , {SCALER_FILE}")

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  2 of 2 completed



=== Run 1/5 (seed=101) ===


/Users/derek/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/Users/derek/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


[Final Fold] Cov=41.0%  AvgUp=0.2060%
  → Saved best model to best_model_FBK.pth

=== Run 2/5 (seed=102) ===


...

[Final Fold] Cov=48.6%  AvgUp=-0.0071%

=== Run 4/5 (seed=104) ===


/Users/derek/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


[Final Fold] Cov=46.7%  AvgUp=0.1019%

=== Run 5/5 (seed=105) ===


/Users/derek/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


[Final Fold] Cov=45.4%  AvgUp=0.1689%

Pooled final-fold block-bootstrap p-value = 0.0969
Best run #2  AvgUp = 0.2415%
Saved files →  best_model_FBK.pth , scaler_FBK.pkl
